In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install git+https://github.com/ChaoningZhang/MobileSAM.git
!mkdir -p weights
!wget -nc https://github.com/ChaoningZhang/MobileSAM/raw/master/weights/mobile_sam.pt -P ./weights/

  Cloning https://github.com/ChaoningZhang/MobileSAM.git to /tmp/pip-req-build-4p81gt_h
  Running command git clone --filter=blob:none --quiet https://github.com/ChaoningZhang/MobileSAM.git /tmp/pip-req-build-4p81gt_h
  Resolved https://github.com/ChaoningZhang/MobileSAM.git to commit b01a9ccef3b9e10b099b544efe004d0871802c3b
  Preparing metadata (setup.py) ... done
  Created wheel for mobile_sam: filename=mobile_sam-1.0-py3-none-any.whl size=42431 sha256=723d5382709f6a6fc5255a8c36b7812d425663ea8801dba0d280dc13f7cd648a
  Stored in directory: /tmp/pip-ephem-wheel-cache-sx8b78x9/wheels/5f/88/d6/5c0b5d4d64a06e19190d50269d8725c8aeadb128c966801af5
Successfully built mobile_sam
--2026-03-12 15:52:50--  https://github.com/ChaoningZhang/MobileSAM/raw/master/weights/mobile_sam.pt
Resolving github.com (github.com)... 140.82.114.4
Connecting to github.com (github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/Chaon

In [ ]:
!pip install peft

In [ ]:
import os
import cv2
import ast
import pandas as pd
import numpy as np
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset


from peft import LoraConfig, get_peft_model, PeftModel
from mobile_sam import sam_model_registry

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.12/dist-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/usr/local/lib/python3.12/dist-packages/mobile_sam/modeling/tiny_vit_sam.py:656: UserWarning: Overwriting tiny_vit_5m_224 in registry with mobile_sam.modeling.tiny_vit_sam.tiny_vit_5m_224. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  return register_model(fn_wrapper)
/usr/local/lib/python3.12/dist-packages/mobile_sam/modeling/tiny_vit_sam.py:656: UserWarning: Overwriting tiny_vit_11m

## Custom Dataset classes

There are two custom dataset classes defined here. Please choose the one which is suitable for your use case.

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

### SegmentationDataset class

This class is for the dataset where it contains the ground truth segmentation masks for bounding box(es) extraction. Just need to pass the paths that contain the images and ground truth masks. It will automatically prepare the bounding box(es) coordinates from the ground truth masks.

In [ ]:
class SegmentationDataset(Dataset):
    def __init__(self, image_dir, mask_dir, img_size=(1024, 1024), mean=IMAGENET_MEAN, std=IMAGENET_STD):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.images = sorted(os.listdir(image_dir))
        self.masks = sorted(os.listdir(mask_dir))
        self.img_size = img_size
        self.mean = np.array(mean)
        self.std = np.array(std)

    def __len__(self):
        return len(self.images)

    def get_bounding_boxes_from_mask(self, mask, padding_factor=0.1, min_area_threshold=5):
        """Get one or multiple bounding boxes from binary mask."""
        _, binary_mask = cv2.threshold(mask, 1, 255, cv2.THRESH_BINARY)
        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary_mask, 8, cv2.CV_32S)

        boxes = []
        for i in range(1, num_labels):
            x, y, w, h, area = (
                stats[i, cv2.CC_STAT_LEFT],
                stats[i, cv2.CC_STAT_TOP],
                stats[i, cv2.CC_STAT_WIDTH],
                stats[i, cv2.CC_STAT_HEIGHT],
                stats[i, cv2.CC_STAT_AREA],
            )
            if area < min_area_threshold:
                continue

            pad = int(max(w, h) * padding_factor)
            x_min, y_min = max(0, x - pad), max(0, y - pad)
            x_max, y_max = min(mask.shape[1], x + w + pad), min(mask.shape[0], y + h + pad)
            boxes.append([x_min, y_min, x_max, y_max])

        return boxes

    def __getitem__(self, idx):
        img_name = self.images[idx]
        mask_name = self.masks[idx]

        image_path = os.path.join(self.image_dir, img_name)
        mask_path = os.path.join(self.mask_dir, mask_name)

        # Load image and mask
        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        original_H, original_W = mask.shape
        boxes = self.get_bounding_boxes_from_mask(mask)

        # --- Resize with Padding (Letterbox) ---
        target_H, target_W = self.img_size
        scale = min(target_W / original_W, target_H / original_H)
        new_W, new_H = int(original_W * scale), int(original_H * scale)
        image_resized = cv2.resize(image, (new_W, new_H))
        mask_resized = cv2.resize(mask, (new_W, new_H), interpolation=cv2.INTER_NEAREST)

        # Create padded image and mask (black background)
        padded_image = np.zeros((target_H, target_W, 3), dtype=np.uint8)
        padded_mask = np.zeros((target_H, target_W), dtype=np.uint8)
        padded_image[:new_H, :new_W, :] = image_resized
        padded_mask[:new_H, :new_W] = mask_resized

        # Rescale bounding boxes
        boxes_rescaled = []
        for box in boxes:
            x_min, y_min, x_max, y_max = box
            boxes_rescaled.append([
                x_min * scale,
                y_min * scale,
                x_max * scale,
                y_max * scale
            ])

        # Image normalization
        norm_image = padded_image.astype("float32") / 255.0
        norm_image = (norm_image - self.mean) / self.std
        image_tensor = torch.from_numpy(norm_image).permute(2, 0, 1)

        mask_tensor = torch.from_numpy((padded_mask > 0).astype(np.float32)).unsqueeze(0)

        # Return all boxes, not just one (SAM can handle multiple)
        return {
            "image": image_tensor,
            "mask": mask_tensor,
            "bboxes": torch.tensor(boxes_rescaled, dtype=torch.float32),
            "image_name": img_name,
            "original_size": torch.tensor([original_H, original_W]),
            "scale": torch.tensor(scale)
        }

**Note:** Please modify the paths below to your own paths that contain the wound images and their ground truth masks

In [ ]:
dataset_images_path = "/content/drive/MyDrive/Wound_Assessment/Dataset/data_wound_seg/train_images/"
dataset_masks_path = "/content/drive/MyDrive/Wound_Assessment/Dataset/data_wound_seg/train_masks/"

In [ ]:
# dataset_images_path = [
#     "/content/drive/MyDrive/Wound_Assessment/Dataset/data_wound_seg/train_images",
#     "/content/drive/MyDrive/Wound_Assessment/Dataset/data_wound_seg/test_images"
# ]

# dataset_masks_path = [
#     "/content/drive/MyDrive/Wound_Assessment/Dataset/data_wound_seg/train_masks",
#     "/content/drive/MyDrive/Wound_Assessment/Dataset/data_wound_seg/test_masks"
# ]

In [ ]:
dataset = SegmentationDataset(dataset_images_path, dataset_masks_path)

### RealWorldInferenceDataset class
This class is for real world collected dataset that has no ground truth segmentation masks for bounding box(es) extraction. Therefore, we need to manually annotate the bounding box(es) of the wound image in PASCAL format (x1, y1, x2, y2) and record them in a csv file.

About the csv file, it must contain two columns: **image_name** and **bbox**. If you have different column names, you can also modify the code below to match the column names in your csv file. Please ensure your image_name is matched and the bounding box is in the correct format: [[x1, y1, x2, y2]]. If there are two bounding boxes, the format is [[x1, y1, x2, y2], [x1, y1, x2, y2]].

In [ ]:
class RealWorldInferenceDataset(Dataset):
    def __init__(self, image_dir, csv_path, img_size=(1024, 1024), mean=IMAGENET_MEAN, std=IMAGENET_STD):
        self.image_dir = image_dir
        self.img_size = img_size
        self.mean = np.array(mean)
        self.std = np.array(std)

        # 1. Load CSV
        # We assume the CSV has columns: 'image_name' and 'bbox'
        df = pd.read_csv(csv_path)

        # Create a dictionary for fast lookup:
        # {'image_01.jpg': "[[10, 10, 100, 100]]", ...}
        self.bbox_map = dict(zip(df['image_name'], df['bbox']))

        # 2. Filter images
        # Only include images that exist in BOTH the folder AND the CSV
        available_files = set(os.listdir(image_dir))
        self.images = [img for img in df['image_name'] if img in available_files]

        print(f"Found {len(self.images)} images with matching bounding boxes in CSV.")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        image_path = os.path.join(self.image_dir, img_name)

        # Load Image
        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Get original dimensions
        original_H, original_W = image.shape[:2]

        # Get Bounding Boxes
        # The CSV likely stores them as strings, so we parse them into lists
        boxes_raw = self.bbox_map[img_name]
        if isinstance(boxes_raw, str):
            boxes_raw = ast.literal_eval(boxes_raw)

        # --- Resize with Padding (Letterbox) ---
        target_H, target_W = self.img_size
        # Calculate scale to fit within target size maintaining aspect ratio
        scale = min(target_W / original_W, target_H / original_H)
        new_W, new_H = int(original_W * scale), int(original_H * scale)

        image_resized = cv2.resize(image, (new_W, new_H))

        # Create padded image (black background)
        padded_image = np.zeros((target_H, target_W, 3), dtype=np.uint8)
        # Paste resized image at (0,0)
        padded_image[:new_H, :new_W, :] = image_resized

        # --- Rescale Bounding Boxes ---
        # If we resize the image, we MUST resize the box coordinates too
        boxes_rescaled = []
        for box in boxes_raw:
            x_min, y_min, x_max, y_max = box
            boxes_rescaled.append([
                x_min * scale,
                y_min * scale,
                x_max * scale,
                y_max * scale
            ])

        # Normalize Image
        image_tensor = padded_image.astype("float32") / 255.0
        image_tensor = (image_tensor - self.mean) / self.std
        image_tensor = torch.from_numpy(image_tensor).permute(2, 0, 1)

        # Return dict with extra info for reconstruction
        return {
            "image": image_tensor,
            "bboxes": torch.tensor(boxes_rescaled, dtype=torch.float32),
            "image_name": img_name,
            "original_size": torch.tensor([original_H, original_W]),
            "scale": torch.tensor(scale)
        }

**Note:** Please modify the paths below to the your own paths that contain real world dataset and the csv file.

In [ ]:
real_world_dataset_path = "/content/drive/MyDrive/Wound_Assessment/Dataset/data_wound_seg/SSL_manual_seg/"
csv_file_path = "/content/drive/MyDrive/Wound_Assessment/bbox_for_SSL.csv"

In [ ]:
dataset = RealWorldInferenceDataset(real_world_dataset_path, csv_file_path)

Found 12 images with matching bounding boxes in CSV.


## Model Initialization

### MobileSAM

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Load the MobileSAM model
model_type = "vit_t"
checkpoint_path = "./weights/mobile_sam.pt"

mobile_sam = sam_model_registry[model_type](checkpoint=checkpoint_path)

In [ ]:
# Load trained mask decoder
decoder_path = "/content/drive/MyDrive/Wound_Assessment/mask_decoder.pth"

# Capture the loading message
msg = mobile_sam.mask_decoder.load_state_dict(torch.load(decoder_path, map_location=device))

# Check for issues
if len(msg.missing_keys) == 0 and len(msg.unexpected_keys) == 0:
    print("Mask Decoder: All weights loaded successfully with no mismatches.")
else:
    print("Mask Decoder Load Warning:")
    print(f"  Missing Keys: {msg.missing_keys}")
    print(f"  Unexpected Keys: {msg.unexpected_keys}")

Mask Decoder: All weights loaded successfully with no mismatches.


In [ ]:
# Load LoRA into the image encoder
lora_path = "/content/drive/MyDrive/Wound_Assessment/lora_image_encoder"
mobile_sam.image_encoder = PeftModel.from_pretrained(mobile_sam.image_encoder, lora_path)

# Verification Steps:
# 1. Check if an adapter is active
active_adapters = mobile_sam.image_encoder.active_adapters
print(f"Active LoRA Adapters: {active_adapters}")

# 2. Check for missing keys (PEFT models often warn during .from_pretrained)
# If you want to be 100% sure, check if the lora layers exist in the modules
has_lora = any("lora_" in name for name, _ in mobile_sam.image_encoder.named_modules())
if has_lora:
    print("LoRA layers detected in the Image Encoder.")
else:
    print("ERROR: No LoRA layers found. The adapter was not applied correctly.")

# 3. Print trainable parameters (should be 0 for inference, but confirms structure)
mobile_sam.image_encoder.print_trainable_parameters()

Active LoRA Adapters: ['default']
LoRA layers detected in the Image Encoder.
trainable params: 0 || all params: 6,184,316 || trainable%: 0.0000


In [ ]:
mobile_sam.to(device)

Sam(
  (image_encoder): PeftModel(
    (base_model): LoraModel(
      (model): TinyViT(
        (patch_embed): PatchEmbed(
          (seq): Sequential(
            (0): Conv2d_BN(
              (c): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
              (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            )
            (1): GELU(approximate='none')
            (2): Conv2d_BN(
              (c): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
              (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            )
          )
        )
        (layers): ModuleList(
          (0): ConvLayer(
            (blocks): ModuleList(
              (0-1): 2 x MBConv(
                (conv1): Conv2d_BN(
                  (c): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
                  (bn): BatchNorm2d(256, eps=1e-05, momentum=0

### Finetuner Wrapper Class

In [ ]:
class MobileSAMFineTuner(nn.Module):
    def __init__(self, sam_model):
        super().__init__()
        self.sam = sam_model

    def forward(self, images: torch.Tensor, bboxes: list):
        # images: [B, 3, 1024, 1024]
        # bboxes: List of tensors, where bboxes[i] is [N_boxes, 4]

        _, _, H, W = images.shape

        # 1. Compute Image Embeddings (Run once per image)
        image_embeddings = self.sam.image_encoder(images)
        dense_pe = self.sam.prompt_encoder.get_dense_pe()

        # Prepare lists to match the "Previous Wrapper" return format
        final_masks_list = []
        iou_preds_list = []

        B = len(bboxes)

        for i in range(B):
            curr_box = bboxes[i] # Shape [N, 4]

            # Safety check for images with no boxes
            if curr_box.shape[0] == 0:
                 # Return empty tensors so the list index stays aligned
                 final_masks_list.append(torch.zeros(0, 1, H, W, device=images.device))
                 iou_preds_list.append(torch.zeros(0, 1, device=images.device))
                 continue

            curr_embedding = image_embeddings[i].unsqueeze(0)

            # Prompt encoder (Handles N boxes)
            sparse_embeddings, dense_embeddings = self.sam.prompt_encoder(
                points=None,
                boxes=curr_box,
                masks=None,
            )

            # Mask decoder
            low_res_masks, iou_predictions = self.sam.mask_decoder(
                image_embeddings=curr_embedding,
                image_pe=dense_pe,
                sparse_prompt_embeddings=sparse_embeddings,
                dense_prompt_embeddings=dense_embeddings,
                multimask_output=False,
            )
            # low_res_masks shape: [N, 1, 256, 256]

            # Upsample NOW (per image) instead of stacking first
            upsampled_masks = F.interpolate(
                low_res_masks,
                size=(H, W),
                mode="bilinear",
                align_corners=False,
            )
            # upsampled_masks shape: [N, 1, 1024, 1024]

            final_masks_list.append(upsampled_masks)
            iou_preds_list.append(iou_predictions)

        # Return LISTS, not stacked tensors.
        # The evaluation loop will access [0] to get the tensor for the first image.
        return final_masks_list, iou_preds_list

In [ ]:
finetuner = MobileSAMFineTuner(sam_model=mobile_sam)

In [ ]:
finetuner.to(device)
finetuner.eval()

MobileSAMFineTuner(
  (sam): Sam(
    (image_encoder): PeftModel(
      (base_model): LoraModel(
        (model): TinyViT(
          (patch_embed): PatchEmbed(
            (seq): Sequential(
              (0): Conv2d_BN(
                (c): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
                (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
              )
              (1): GELU(approximate='none')
              (2): Conv2d_BN(
                (c): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
                (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
              )
            )
          )
          (layers): ModuleList(
            (0): ConvLayer(
              (blocks): ModuleList(
                (0-1): 2 x MBConv(
                  (conv1): Conv2d_BN(
                    (c): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), b

## Masks Generation & Save

**Note:** Please modify the `output_dir` below to your own desired path.

In [ ]:
# Define where to save the masks
output_dir = "/content/drive/MyDrive/Wound_Assessment/SSL_TSegNet_masks/Manual/"
os.makedirs(output_dir, exist_ok=True)

print(f"Saving to {output_dir}")

Saving to /content/drive/MyDrive/Wound_Assessment/SSL_TSegNet_masks/Manual/


**Note:**
- There is also some padding added (`k_size` of 20 approximately enlarged the wound mask by 10 pixels). Increase `k_size` if you want more padding.
- The mask is resized to (224, 224) already

In [ ]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=False)

with torch.no_grad():
  # Wrap loader with tqdm for a progress bar
  for batch in tqdm(dataloader):
      images = batch["image"].to(device).float()
      bboxes = [batch["bboxes"][0].to(device).float()]
      image_name = batch["image_name"][0]
      original_size = batch["original_size"][0]
      scale = batch["scale"][0].item()

      # Skip if no bounding boxes are detected (SAM needs a prompt)
      if bboxes[0].numel() == 0:
          print(f"Skipping {image_name}: No bounding boxes detected.")
          continue

      # --- Forward pass (Generate Mask) ---
      pred_masks_list, _ = finetuner(images, bboxes)

      # Get the mask for the first image in batch
      pred_masks = pred_masks_list[0]

      # Combine multiple masks (if multiple boxes) into one binary mask
      combined_pred_mask = (torch.sigmoid(pred_masks) > 0.5).float().sum(dim=0, keepdim=True)
      combined_pred_mask = (combined_pred_mask > 0).float()

      # Squeeze to shape [H, W] (matches 1024x1024 input)
      mask_tensor = combined_pred_mask.squeeze()

      # --- Convert to Image Format ---
      mask_np = mask_tensor.cpu().numpy().astype(np.uint8) * 255

      # --- Remove Padding & Resize Back to Original ---
      # 1. Calculate the valid region (where the image was pasted)
      orig_H, orig_W = int(original_size[0].item()), int(original_size[1].item())
      valid_H = int(orig_H * scale)
      valid_W = int(orig_W * scale)

      # 2. Crop the valid region
      mask_cropped = mask_np[:valid_H, :valid_W]

      # 3. Apply Dilation (Optional, but keeps previous behavior of 'enlarging' mask slightly)
      # k_size = 20
      # kernel = np.ones((k_size, k_size), np.uint8)
      # mask_processed = cv2.dilate(mask_cropped, kernel, iterations=1)

      # 3. Dilation by percentage
      padding_percent = 0.05 # Adjust this (0.05 = 5%)
      dynamic_k_size = int(max(mask_cropped.shape) * padding_percent)

      # Ensure k_size is at least 1
      dynamic_k_size = max(1, dynamic_k_size)

      kernel = np.ones((dynamic_k_size, dynamic_k_size), np.uint8)
      mask_processed = cv2.dilate(mask_cropped, kernel, iterations=1)

      # 4. Resize back to original size
      mask_final = cv2.resize(mask_processed, (orig_W, orig_H), interpolation=cv2.INTER_NEAREST)

      # --- Save the Mask ---
      filename_no_ext = os.path.splitext(image_name)[0]
      save_path = os.path.join(output_dir, f"{filename_no_ext}.png")

      cv2.imwrite(save_path, mask_final)
      print(f"Successfully saved the mask to {save_path}\n")

  8%|▊         | 1/12 [00:01<00:20,  1.88s/it]

Successfully saved the mask to /content/drive/MyDrive/Wound_Assessment/SSL_TSegNet_masks/Manual/fusc_0223.png



 17%|█▋        | 2/12 [00:02<00:10,  1.09s/it]

Successfully saved the mask to /content/drive/MyDrive/Wound_Assessment/SSL_TSegNet_masks/Manual/fusc_0502.png



 25%|██▌       | 3/12 [00:03<00:08,  1.08it/s]

...

Successfully saved the mask to /content/drive/MyDrive/Wound_Assessment/SSL_TSegNet_masks/Manual/fusc_0483.png



 92%|█████████▏| 11/12 [00:09<00:00,  1.07it/s]

Successfully saved the mask to /content/drive/MyDrive/Wound_Assessment/SSL_TSegNet_masks/Manual/fusc_0736.png



100%|██████████| 12/12 [00:09<00:00,  1.23it/s]

Successfully saved the mask to /content/drive/MyDrive/Wound_Assessment/SSL_TSegNet_masks/Manual/fusc_0950.png

